# Bronze Layer - Data Ingestion Overview

Surveys every table ingested into **bronze_lakehouse** so you can confirm what landed and inspect it.

| | |
| --- | --- |
| **Lakehouse** | `bronze_lakehouse` |
| **AOI** | Pipeline-selected latitude, longitude, and catalog radius |
| **Layers** | Planetary Computer STAC, DataBC geology, and DataBC soil survey |

Run top-to-bottom. The default lakehouse must be attached. Empty DataBC layers outside British Columbia coverage are displayed as data gaps.

## 1. Discover all bronze tables

In [ ]:
# List every table in the default lakehouse and keep the bronze_ ones
all_tables = [t.name for t in spark.catalog.listTables()]
bronze_tables = sorted([t for t in all_tables if t.startswith("bronze_")])

print(f"Found {len(bronze_tables)} bronze tables:\n")
for t in bronze_tables:
    print(f"  - {t}")

## 2. Row counts per table

In [ ]:
# Build a summary of row counts and column counts for each bronze table
from pyspark.sql import Row

summary = []
for t in bronze_tables:
    try:
        df = spark.table(t)
        summary.append(Row(table=t, rows=df.count(), columns=len(df.columns)))
    except Exception as e:
        summary.append(Row(table=t, rows=-1, columns=-1))
        print(f"  ! {t}: {e}")

summary_df = spark.createDataFrame(summary).orderBy("table")
print(f"Total bronze tables: {len(summary)}")
print(f"Total rows across all bronze tables: {sum(r.rows for r in summary if r.rows > 0)}")
display(summary_df)

## 3. Schema + sample rows for each table

In [ ]:
# Inspect schema and a few sample rows for every bronze table
for t in bronze_tables:
    print("=" * 80)
    print(f"TABLE: {t}")
    print("=" * 80)
    try:
        df = spark.table(t)
        print(f"Rows: {df.count()}  |  Columns: {len(df.columns)}")
        df.printSchema()
        display(df.limit(5))
    except Exception as e:
        print(f"  ! Could not read {t}: {e}")
    print()

- **DataBC vector tables** — geology (quaternary, bedrock, faults) and the BC
  soil survey (SIFT) → drawn as their actual **GeoJSON geometries**.
- **Click any feature** (raster footprint, geology polygon, or soil polygon) to
  see **every column** for that record in a scrollable popup.

In [ ]:
# Pipeline parameters
LATITUDE = 49.2193
LONGITUDE = -122.5984
RADIUS_KM = 20

In [ ]:
# Map helpers: ensure folium, fetch real raster tiles from Planetary Computer,
# and build a "show every column" popup.
import sys, subprocess, json, requests
from urllib.parse import urlparse, parse_qs

try:
    import folium
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "folium", "-q"], check=True)
    import folium

LAT = float(LATITUDE)
LON = float(LONGITUDE)
RADIUS_KM = float(RADIUS_KM)
if not -90.0 <= LAT <= 90.0:
    raise ValueError("LATITUDE must be between -90 and 90 degrees.")
if not -180.0 <= LON <= 180.0:
    raise ValueError("LONGITUDE must be between -180 and 180 degrees.")
if not 0.0 < RADIUS_KM <= 100.0:
    raise ValueError("RADIUS_KM must be greater than 0 and no more than 100 km.")
AOI_LABEL = f"AOI center ({LAT:.4f}, {LON:.4f})"

PC_STAC = "https://planetarycomputer.microsoft.com/api/stac/v1"
PC_ATTR = "Imagery &copy; Microsoft Planetary Computer"
bbox_cols = {"bbox_minx", "bbox_miny", "bbox_maxx", "bbox_maxy"}

# Distinct CSS-safe color per table (valid for Leaflet paths).
palette = [
    "blue", "green", "purple", "orange", "darkred", "teal", "darkgreen",
    "navy", "magenta", "brown", "gray", "steelblue", "crimson", "olive",
]
color_for = {table: palette[index % len(palette)] for index, table in enumerate(bronze_tables)}

# CSS gradients approximating common Planetary Computer continuous colormaps.
COLORMAP_GRADIENTS = {
    "terrain": "linear-gradient(to right,#333366,#1f9b6c,#f2e09a,#b8763d,#ffffff)",
    "viridis": "linear-gradient(to right,#440154,#3b528b,#21918c,#5ec962,#fde725)",
    "magma": "linear-gradient(to right,#000004,#3b0f70,#8c2981,#de4968,#fcfdbf)",
    "inferno": "linear-gradient(to right,#000004,#420a68,#932667,#dd513a,#fcffa4)",
    "plasma": "linear-gradient(to right,#0d0887,#6a00a8,#b12a90,#e16462,#f0f921)",
    "cividis": "linear-gradient(to right,#00224e,#35456c,#666970,#9d9540,#fee838)",
    "gray": "linear-gradient(to right,#000000,#ffffff)",
    "greys": "linear-gradient(to right,#ffffff,#000000)",
    "rdylgn": "linear-gradient(to right,#d73027,#fee08b,#1a9850)",
    "spectral": "linear-gradient(to right,#d53e4f,#fee08b,#99d594,#3288bd)",
    "rdbu": "linear-gradient(to right,#b2182b,#f7f7f7,#2166ac)",
}

# A few items per raster table keeps the live tile fetch fast.
MAX_RASTER_ITEMS = 5
MAX_VECTOR_FEATURES = 1000


def fetch_item(collection, item_id, timeout=30):
    """Fetch a STAC item from Planetary Computer."""
    url = f"{PC_STAC}/collections/{collection}/items/{item_id}"
    response = requests.get(url, timeout=timeout)
    response.raise_for_status()
    return response.json()


def tile_url_for_item(item, timeout=30):
    """Resolve an XYZ raster tile template for a STAC item."""
    tilejson = (item.get("assets", {}) or {}).get("tilejson")
    if not tilejson or not tilejson.get("href"):
        return None
    try:
        document = requests.get(tilejson["href"], timeout=timeout).json()
        tiles = document.get("tiles") or []
        return tiles[0] if tiles else None
    except Exception:
        return None


def preview_url_for_item(item):
    """Return the fallback static PNG preview URL."""
    asset = (item.get("assets", {}) or {}).get("rendered_preview")
    return asset.get("href") if asset else None


def raster_class_entries(item, max_classes=14):
    """Return discrete color and label swatches from STAC classification metadata."""
    output, seen = [], set()
    for asset in (item.get("assets", {}) or {}).values():
        classes = asset.get("classification:classes") or asset.get("classification:bitfields")
        if not classes:
            continue
        for item_class in classes:
            color = item_class.get("color-hint") or item_class.get("color_hint")
            if not color:
                continue
            if not str(color).startswith("#"):
                color = "#" + str(color)
            label = item_class.get("description") or item_class.get("title") or str(item_class.get("value"))
            key = (color.lower(), label)
            if key in seen:
                continue
            seen.add(key)
            output.append((color, label))
        if output:
            break
    extra = max(0, len(output) - max_classes)
    return output[:max_classes], extra


def raster_render_desc(item):
    """Describe the Planetary Computer raster rendering configuration."""
    tilejson = (item.get("assets", {}) or {}).get("tilejson")
    if not tilejson or not tilejson.get("href"):
        return None
    query = parse_qs(urlparse(tilejson["href"]).query)
    colormap = (query.get("colormap_name") or [None])[0]
    if colormap:
        gradient = COLORMAP_GRADIENTS.get(
            colormap.lower(), "linear-gradient(to right,#222,#888,#eee)"
        )
        range_label = ""
        rescale = (query.get("rescale") or [None])[0]
        if rescale:
            parts = rescale.replace("%2C", ",").split(",")
            if len(parts) == 2:
                range_label = f"  ({parts[0]}–{parts[1]})"
        return ("gradient:" + gradient, f"{colormap} colormap{range_label}")
    assets = ",".join(query.get("assets", []))
    if "visual" in assets or "expression" in query:
        return (None, "True/false-color composite (RGB)")
    if assets:
        return (None, f"Render: {assets}")
    return None


def _esc(value):
    text = "" if value is None else str(value)
    if len(text) > 600:
        text = text[:600] + " …"
    return text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")


def popup_html(values, title=None):
    """Build a scrollable HTML table showing every field in a row dictionary."""
    rows = "".join(
        f"<tr><td style='font-weight:600;padding:2px 6px;vertical-align:top;"
        f"white-space:nowrap'>{_esc(key)}</td>"
        f"<td style='padding:2px 6px;word-break:break-word'>{_esc(value)}</td></tr>"
        for key, value in values.items()
    )
    heading = (
        f"<div style='font-weight:700;margin-bottom:4px'>{_esc(title)}</div>"
        if title else ""
    )
    return (
        "<div style='max-height:320px;max-width:440px;overflow:auto;"
        "font-family:system-ui,sans-serif;font-size:12px'>"
        + heading + "<table>" + rows + "</table></div>"
    )


def add_legend(map_object, title, entries):
    """Add a fixed-position legend box to a folium map."""
    items = []
    for swatch, label in entries:
        if swatch is None:
            glyph = "<span style='display:inline-block;width:14px;margin-right:6px'></span>"
        elif isinstance(swatch, str) and swatch.startswith("marker:"):
            color = swatch.split(":", 1)[1]
            glyph = (
                f"<span style='color:{color};margin-right:6px;"
                f"font-size:14px;line-height:14px'>&#9733;</span>"
            )
        elif isinstance(swatch, str) and swatch.startswith("ring:"):
            color = swatch.split(":", 1)[1]
            glyph = (
                f"<span style='display:inline-block;width:13px;height:13px;"
                f"border:2px solid {color};border-radius:50%;margin-right:6px'></span>"
            )
        elif isinstance(swatch, str) and swatch.startswith("gradient:"):
            gradient = swatch.split(":", 1)[1]
            glyph = (
                f"<span style='display:inline-block;width:34px;height:13px;"
                f"background:{gradient};border:1px solid #555;margin-right:6px'></span>"
            )
        else:
            glyph = (
                f"<span style='display:inline-block;width:14px;height:14px;"
                f"background:{swatch};border:1px solid #555;margin-right:6px'></span>"
            )
        items.append(
            f"<div style='display:flex;align-items:center;margin:2px 0'>"
            f"{glyph}<span>{_esc(label)}</span></div>"
        )
    html = (
        "<div style='position:fixed;bottom:22px;left:22px;z-index:9999;"
        "background:rgba(255,255,255,0.93);padding:10px 12px;border:1px solid #888;"
        "border-radius:6px;font-family:system-ui,sans-serif;font-size:12px;"
        "box-shadow:0 1px 5px rgba(0,0,0,0.3);max-width:260px;"
        "max-height:60vh;overflow:auto'>"
        f"<div style='font-weight:700;margin-bottom:5px'>{_esc(title)}</div>"
        + "".join(items) + "</div>"
    )
    map_object.get_root().html.add_child(folium.Element(html))


print(f"Map helpers ready for {AOI_LABEL}. Raster items per table: {MAX_RASTER_ITEMS}")

In [ ]:
# render_layer(table): draw one bronze layer on its own map, with a legend.
# Real raster tiles for STAC tables, GeoJSON for vector tables, and full popups.

def new_map():
    map_object = folium.Map(location=[LAT, LON], zoom_start=11, tiles="CartoDB positron")
    folium.Marker(
        [LAT, LON], tooltip=AOI_LABEL,
        icon=folium.Icon(color="red", icon="star"),
    ).add_to(map_object)
    folium.Circle(
        [LAT, LON], radius=RADIUS_KM * 1000, color="red",
        fill=False, weight=2, tooltip=f"{RADIUS_KM:g} km catalog AOI",
    ).add_to(map_object)
    return map_object


def render_layer(table_name):
    """Render one bronze table as a map with a legend."""
    if table_name not in set(bronze_tables):
        print(f"  - {table_name}: not found in bronze tables — skipped")
        return
    color = color_for.get(table_name, "blue")
    try:
        dataframe = spark.table(table_name)
    except Exception as error:
        print(f"  ! skip {table_name}: {error}")
        return
    columns = set(dataframe.columns)
    map_object = new_map()
    rendered = 0
    legend = [
        ("marker:red", AOI_LABEL),
        ("ring:red", f"{RADIUS_KM:g} km catalog AOI radius"),
    ]

    if bbox_cols.issubset(columns):
        rows = dataframe.limit(MAX_RASTER_ITEMS).collect()
        class_entries, class_extra = [], 0
        render_description = None
        for row in rows:
            values = row.asDict()
            minx, miny = values.get("bbox_minx"), values.get("bbox_miny")
            maxx, maxy = values.get("bbox_maxx"), values.get("bbox_maxy")
            item_id = values.get("item_id")
            collection = values.get("collection")

            tile_url = preview_url = None
            if item_id and collection:
                try:
                    item = fetch_item(collection, item_id)
                    tile_url = tile_url_for_item(item)
                    preview_url = preview_url_for_item(item)
                    if not class_entries:
                        class_entries, class_extra = raster_class_entries(item)
                    if render_description is None:
                        render_description = raster_render_desc(item)
                except Exception as error:
                    print(f"    ! {table_name} {item_id}: {error.__class__.__name__}")

            if tile_url:
                folium.TileLayer(
                    tiles=tile_url,
                    attr=PC_ATTR,
                    name=str(item_id),
                    overlay=True,
                    control=True,
                    opacity=0.9,
                ).add_to(map_object)
                rendered += 1
            elif preview_url and None not in (minx, miny, maxx, maxy):
                folium.raster_layers.ImageOverlay(
                    image=preview_url,
                    bounds=[[miny, minx], [maxy, maxx]],
                    opacity=0.9,
                    name=str(item_id),
                ).add_to(map_object)
                rendered += 1

            if None not in (minx, miny, maxx, maxy):
                folium.Rectangle(
                    bounds=[[miny, minx], [maxy, maxx]],
                    color=color,
                    weight=2,
                    fill=False,
                    popup=folium.Popup(popup_html(values, title=item_id), max_width=460),
                    tooltip=str(item_id or table_name),
                ).add_to(map_object)
        legend.append(("ring:" + color, "Item footprint (click for details)"))
        if class_entries:
            legend.append((None, "Raster classes:"))
            legend.extend(class_entries)
            if class_extra:
                legend.append((None, f"… +{class_extra} more class(es)"))
        elif render_description:
            legend.append((None, "Raster coloring:"))
            legend.append(render_description)
        else:
            legend.append((color, "Live raster tiles (toggle top-right)"))
        label = f"{rendered} raster item(s)"

    elif "geometry_json" in columns:
        rows = dataframe.limit(MAX_VECTOR_FEATURES).collect()
        for row in rows:
            values = row.asDict()
            geometry_json = values.get("geometry_json")
            if not geometry_json:
                continue
            try:
                geometry = json.loads(geometry_json)
            except Exception:
                continue
            attributes = {key: value for key, value in values.items() if key != "geometry_json"}
            folium.GeoJson(
                geometry,
                style_function=lambda _, selected_color=color: {
                    "color": selected_color,
                    "weight": 2,
                    "fillColor": selected_color,
                    "fillOpacity": 0.25,
                },
                popup=folium.Popup(popup_html(attributes, title=table_name), max_width=460),
                tooltip=table_name,
            ).add_to(map_object)
            rendered += 1
        legend.append((color, "Vector features (click for details)"))
        label = f"{rendered} geometry feature(s)"

    else:
        print(f"  - {table_name}: no spatial columns (bbox_* or geometry_json) — skipped")
        return

    folium.LayerControl(collapsed=False).add_to(map_object)
    add_legend(map_object, table_name, legend)
    print(f"  + {table_name}: {label} ({color})")

    header = (
        f"<h3 style='font-family:system-ui,sans-serif;margin:14px 0 4px'>"
        f"{table_name} <span style='font-weight:400;color:#666;font-size:13px'>"
        f"— {label}</span></h3>"
    )
    try:
        displayHTML(header + map_object._repr_html_())
    except NameError:
        display(map_object)


print("render_layer() ready — one map per cell below, each with a legend.")
print("Tables available:", ", ".join(bronze_tables))

In [ ]:
# Sentinel-2 L2A — optical imagery
render_layer("bronze_sentinel_2_l2a")

In [ ]:
# Sentinel-1 RTC — radar (SAR)
render_layer("bronze_sentinel_1_rtc")

In [ ]:
# ALOS PALSAR mosaic — L-band radar
render_layer("bronze_alos_palsar_mosaic")

In [ ]:
# Copernicus DEM GLO-30 — 30 m elevation
render_layer("bronze_cop_dem_glo_30")

In [ ]:
# ESA WorldCover — land cover
render_layer("bronze_esa_worldcover")

In [ ]:
# IO LULC 9-class — land use / land cover
render_layer("bronze_io_lulc_9_class")

In [ ]:
# HGB — above/below-ground biomass
render_layer("bronze_hgb")

In [ ]:
# BC Quaternary / surficial geology
render_layer("bronze_bc_quaternary_geology")

In [ ]:
# BC bedrock geology
render_layer("bronze_bc_bedrock_geology")

In [ ]:
# BC geological faults
render_layer("bronze_bc_geological_faults")

In [ ]:
# BC soil survey (Soil Information Finder Tool / SIFT)
render_layer("bronze_bc_soil_survey_polygons")